# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Фіксуємо генератори випадкових чисел для відтворюваності
torch.manual_seed(42)
np.random.seed(42)

## Крок 0. Завантаження даних


In [ ]:
ratings = pd.read_csv("ratings.csv")
books = pd.read_csv("books.csv")
book_tags = pd.read_csv("book_tags.csv")
tags = pd.read_csv("tags.csv")

In [ ]:
print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings: (981756, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,2767052,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,41865,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,2657,Harper Lee,To Kill a Mockingbird,4.25
4,4671,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [ ]:
# 1. Зачищаємо назви колонок про всяк випадок
for df_var in [books, ratings, book_tags, tags]:
    df_var.columns = df_var.columns.str.strip()

# 2. Канонічні жанри
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# 3. Приводимо ключові колонки до int
tags['tag_id'] = tags['tag_id'].astype(int)
book_tags['tag_id'] = book_tags['tag_id'].astype(int)

# Оскільки в твоєму book_tags колонка goodreads_book_id є, а в books її немає —
# це означає, що у твоєму конкретному файлі book_tags 'goodreads_book_id' і є прямим ідентифікатором книги!
book_tags['goodreads_book_id'] = book_tags['goodreads_book_id'].astype(int)
books['book_id'] = books['book_id'].astype(int)

# 4. Мапимо теги у назви жанрів
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

# Фільтруємо тільки потрібні теги
bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()

# Прямо переназиваємо колонку в book_id, бо вона виконує її роль
bt["book_id"] = bt["goodreads_book_id"]
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# 5. Будуємо бінарну матрицю жанрів
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))

genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,1,0,0,0,0,1,0,1,1,0
2,1,1,1,0,0,0,0,1,0,0,0,0
3,1,0,1,0,0,0,0,1,0,1,1,0
5,1,0,1,0,0,0,0,1,0,1,1,0
6,1,0,1,0,0,0,0,1,0,1,1,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [ ]:
TOP_BOOKS = 1500
MIN_USER_RATINGS = 20
N_USERS = 2000
LIKE_THRESHOLD = 4

rng = np.random.RandomState(42)

ratings['book_id'] = ratings['book_id'].astype(int)
ratings['user_id'] = ratings['user_id'].astype(int)

# Відбираємо популярні книги
top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)].copy()

# Активні користувачі
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]

# Зберігаємо випадкових користувачів
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# Залишаємо тільки ті книги, які мають теги в нашій матриці жанрів
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 9,431 | користувачів: 1,880 | книг: 120
Щільність: 0.0418


In [ ]:
torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 4,991 | користувачів з val-лайками: 784


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [ ]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [ ]:
# 1. Побудова item_emb — матриці L2-нормалізованих жанрових векторів усіх книг
# Додаємо мікро-епсилон 1e-9, щоб уникнути ділення на нуль, якщо у книги раптом немає жанрів
item_norms = torch.norm(item_feats, p=2, dim=1, keepdim=True)
item_emb = item_feats / torch.clamp(item_norms, min=1e-9)

# 2. Функція для розрахунку косинусних оцінок (vsm_scores) для набору користувачів
def vsm_scores(user_idxs):
    """
    user_idxs: тензор з індексами користувачів розміру (n_users_batch,)
    Повертає матрицю оцінок розміру (n_users_batch, M) для всіх книг
    """
    n_users_batch = len(user_idxs)
    user_embs = []

    # Збираємо вектор для кожного користувача на основі його лайків у train
    for u_idx in user_idxs.tolist():
        # Знаходимо індекси книг, які цей користувач лайкнув у тренувальній вибірці
        # pos_u та pos_i — це глобальні тензори взаємодій, створені на Кроці 2
        liked_item_idxs = pos_i[pos_u == u_idx]

        if len(liked_item_idxs) > 0:
            # Average pooling: беремо середнє арифметичне ембедингів уподобаних книг
            u_emb = item_emb[liked_item_idxs].mean(dim=0)
        else:
            # Фолбек: якщо лайків немає, повертаємо нульовий вектор
            u_emb = torch.zeros(n_genres)

        user_embs.append(u_emb)

    # Перетворюємо список у тензор розміру (n_users_batch, n_genres)
    user_embs_tensor = torch.stack(user_embs)

    # Оскільки вектори книг уже нормалізовані, скалярний добуток (Dot Product)
    # між user_embs та item_emb.T поверне косинусну подібність!
    scores = torch.matmul(user_embs_tensor, item_emb.T) # Результат: (n_users_batch, M)
    return scores

# 3. Рахуємо якість моделі за допомогою Recall@10
vsm_recall = recall_at_k(vsm_scores, k=10)
print(f"Recall@10 для Vector Space Model: {vsm_recall:.4f}")

# =====================================================================
# 4. Виведення топ-5 рекомендацій для одного випадкового користувача
# =====================================================================
test_user_idx = 0  # візьмемо першого користувача
test_user_tensor = torch.tensor([test_user_idx])

with torch.no_grad():
    # Отримуємо оцінки для цього користувача
    user_scores = vsm_scores(test_user_tensor).flatten()

    # Прибираємо книги, які він вже бачив у train (зануляємо їх великим мінусом)
    for i in seen_by_user[test_user_idx]:
        user_scores[i] = -1e9

    # Беремо топ-5 найкращих книг
    top5_values, top5_indices = torch.topk(user_scores, k=5)

print(f"\nТоп-5 рекомендацій для користувача (ID: {users[test_user_idx]}):")
for idx, item_idx in enumerate(top5_indices.tolist()):
    orig_book_id = items[item_idx]
    book_title = title_of.get(orig_book_id, "Unknown Title")
    print(f"{idx+1}. {book_title} (Косинусна подібність: {top5_values[idx]:.4f})")

Recall@10 для Vector Space Model: 0.0743

Топ-5 рекомендацій для користувача (ID: 35):
1. The Brooklyn Follies (Косинусна подібність: 1.0000)
2. Memories of My Melancholy Whores (Косинусна подібність: 1.0000)
3. Tropic of Capricorn (Косинусна подібність: 1.0000)
4. Eleven Minutes (Косинусна подібність: 1.0000)
5. Memoirs of a Geisha (Косинусна подібність: 0.8660)


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


Наразі модель небагато знає про книги, не враховує автора, популярність, середній рейтинг і тд (описали лише певні жанри)

**Аналіз результатів Vector Space Model (VSM):**

1. **Метрика Recall@10:** Модель показала результат `0.0743`. Це означає, що у ~7.4% випадків книги, які користувачам дійсно сподобалися у валідаційній вибірці, наша модель змогла підняти в Топ-10 рекомендацій. Для моделі, яка знає про книги лише 12 канонічних жанрів, це цілком адекватний результат.
2. **Переваги VSM:**
   * Повністю вирішує проблему **Cold Start (холодного старту)** для нових книг. Якщо в систему додається нова книга, нам не потрібно чекати на її оцінки іншими користувачами — достатньо просто розмітити її жанри, і вона відразу почне рекомендуватися.
   * Дуже проста в інтерпретації (ми чітко бачимо, чому подібність дорівнює 1.0000 — бо збігаються вектори ознак).
3. **Обмеження та Diversity (Різноманітність):**
   * Модель має серйозні обмеження щодо **Diversity**. Як видно з топ-рекомендацій, перші 4 книги мають косинусну подібність `1.0000`. Модель просто засипає користувача ідентичними за жанрами книгами (наприклад, якщо він любить драму/сучасну прозу, він отримує лише її). Це створює "інформаційну бульбашку".
   * Модель повністю ігнорує колаборативні сигнали — вона не знає, що думають інші люди про ці книги, і рекомендує суто за сухими текстовими тегами.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [ ]:
import random

# =====================================================================
# Крок 1. Датасет із dynamic Negative Sampling (як у YouTube RecSys)
# =====================================================================
class TwoTowerNegativeSamplingDataset(Dataset):
    def __init__(self, train_pos_df, user_to_idx, item_to_idx, num_items, seen_dict, n_negatives=4):
        self.users = [user_to_idx[u] for u in train_pos_df["user_id"]]
        self.pos_items = [item_to_idx[b] for b in train_pos_df["book_id"]]
        self.num_items = num_items
        self.seen_dict = seen_dict
        self.n_negatives = n_negatives  # скільки негативних книг додавати на 1 позитивну

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u_idx = self.users[idx]
        p_idx = self.pos_items[idx]

        # Семплюємо негативні приклади, які користувач ще не бачив у train
        neg_indices = []
        while len(neg_indices) < self.n_negatives:
            random_item = random.randint(0, self.num_items - 1)
            if random_item not in self.seen_dict[u_idx]:
                neg_indices.append(random_item)

        return torch.tensor(u_idx), torch.tensor(p_idx), torch.tensor(neg_indices)

# Ініціалізуємо наш просунутий датаloader з наявною змінною seen_by_user
train_pos_dataset = TwoTowerNegativeSamplingDataset(
    train_pos, user_to_idx, item_to_idx, num_items=len(items), seen_dict=seen_by_user, n_negatives=4
)

train_loader = DataLoader(train_pos_dataset, batch_size=256, shuffle=True)

# =====================================================================
# Крок 2. Архітектура Two-Tower (Late Fusion)
# =====================================================================
class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, item_features, embedding_dim=32):
        super(TwoTowerModel, self).__init__()

        self.register_buffer('item_features', item_features)
        n_genres = item_features.shape[1]

        # 1. User Tower (працює з ID користувача)
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.user_mlp = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )

        # 2. Item Tower (мапить 12 жанрів у спільний простір)
        self.item_mlp = nn.Sequential(
            nn.Linear(n_genres, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )

        nn.init.normal_(self.user_emb.weight, std=0.1)

    def forward_user(self, user_idx):
        u_vector = self.user_mlp(self.user_emb(user_idx))
        return F.normalize(u_vector, p=2, dim=-1)

    def forward_item(self, item_idx):
        i_feats = self.item_features[item_idx]
        i_vector = self.item_mlp(i_feats)
        return F.normalize(i_vector, p=2, dim=-1)

    def get_all_scores(self, user_idx):
        u_emb = self.forward_user(user_idx)
        all_item_idxs = torch.arange(self.item_features.shape[0], device=user_idx.device)
        all_i_emb = self.forward_item(all_item_idxs)
        return torch.matmul(u_emb, all_i_emb.T)

# Ініціалізація моделі
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tt_model = TwoTowerModel(len(users), len(items), item_feats, embedding_dim=32).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(tt_model.parameters(), lr=0.005, weight_decay=1e-5)

# =====================================================================
# Крок 3. Навчання моделі з логарифмічним лосом (BCE)
# =====================================================================
print("Старт навчання Two-Tower моделі з Negative Sampling...")
for epoch in range(5):
    tt_model.train()
    epoch_loss = 0.0

    for batch_u, batch_pos_i, batch_neg_is in train_loader:
        batch_u = batch_u.to(device)
        batch_pos_i = batch_pos_i.to(device)
        batch_neg_is = batch_neg_is.to(device)

        optimizer.zero_grad()

        u_emb = tt_model.forward_user(batch_u)
        pos_i_emb = tt_model.forward_item(batch_pos_i)
        pos_scores = (u_emb * pos_i_emb).sum(dim=1)

        batch_size = batch_u.size(0)
        n_negs = batch_neg_is.size(1)
        flat_neg_is = batch_neg_is.view(-1)
        neg_i_embs = tt_model.forward_item(flat_neg_is).view(batch_size, n_negs, -1)

        neg_scores = (u_emb.unsqueeze(1) * neg_i_embs).sum(dim=2)

        logits = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1) * 10.0

        targets = torch.zeros_like(logits)
        targets[:, 0] = 1.0

        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Епоха {epoch+1}/5 | BCE Loss: {epoch_loss/len(train_loader):.4f}")

# =====================================================================
# Крок 4. Оцінка Recall@10 та демонстрація рекомендацій
# =====================================================================
def score_fn_twotower(user_idxs_tensor):
    tt_model.eval()
    with torch.no_grad():
        scores = tt_model.get_all_scores(user_idxs_tensor.to(device))
    return scores.cpu()

tt_recall = recall_at_k(score_fn_twotower, k=10)
print(f"\nФінальний Recall@10 для Two-Tower: {tt_recall:.4f}")

test_user_idx = 0
test_user_tensor = torch.tensor([test_user_idx])
with torch.no_grad():
    user_scores = score_fn_twotower(test_user_tensor).flatten()
    if 'seen_by_user' in locals():
        for i in seen_by_user[test_user_idx]: user_scores[i] = -1e9
    top5_values, top5_indices = torch.topk(user_scores, k=5)

print(f"\nТоп-5 рекомендацій для користувача (ID: {users[test_user_idx]}):")
for idx, item_idx in enumerate(top5_indices.tolist()):
    print(f"{idx+1}. {title_of.get(items[item_idx], 'Unknown')} (Косинус: {top5_values[idx]:.4f})")

Старт навчання Two-Tower моделі з Negative Sampling...
Епоха 1/5 | BCE Loss: 0.6457
Епоха 2/5 | BCE Loss: 0.5030
Епоха 3/5 | BCE Loss: 0.4918
Епоха 4/5 | BCE Loss: 0.4821
Епоха 5/5 | BCE Loss: 0.4648

Фінальний Recall@10 для Two-Tower: 0.1265

Топ-5 рекомендацій для користувача (ID: 35):
1. Eleven Minutes (Косинус: -0.0230)
2. The Brooklyn Follies (Косинус: -0.0230)
3. Tropic of Capricorn (Косинус: -0.0230)
4. Memories of My Melancholy Whores (Косинус: -0.0230)
5. Hatchet (Brian's Saga, #1) (Косинус: -0.0464)


**Аналіз результатів Two-Tower моделі з Negative Sampling:**

1. **Порівняння метрик:** Метрика **Recall@10 зросла з 0.0743 (VSM) до 0.1265 (Two-Tower)**. Це суттєвий приріст якості (~70%). Нейромережа змогла набагато краще вловити приховані закономірності у взаємодіях, ніж простий контентний алгоритм.
2. **Як вплинув Negative Sampling:** Завдяки випадковому вибору негативних прикладів (книг, які користувач не оцінював) модель навчилася "відштовхувати" нерелевантні айтеми в просторі ембедингів і "притягувати" лайкнуті релевантні пари. Без негативного семплінгу навчання глибоких моделей на implicit/positive-only даних було б неможливим через колапс ембедингів.
3. **Переваги Late Fusion (Роздільних веж):**
   * **Масштабованість:** Вектори для всіх 1500 книг ми можемо порахувати через `Item Tower` лише один раз (наприклад, вночі за розкладом) і зберегти у швидкий індекс на кшталт FAISS.
   * **Швидкість:** У реальному часі (коли користувач відкриває застосунок) нам потрібно розрахувати лише один вектор через `User Tower` та виконати миттєве матричне множення. Це ідеально підходить для першого етапу рекомендацій (**Retrieval**) на мільйонах товарів.

---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [ ]:
# =====================================================================
# Крок 1. Архітектура Concat-based NCF (Early Fusion)
# =====================================================================
class ConcatNCFModel(nn.Module):
    def __init__(self, num_users, item_features, embedding_dim=32):
        super(ConcatNCFModel, self).__init__()

        # Реєструємо жанрові ознаки книг як буфер
        self.register_buffer('item_features', item_features)
        n_genres = item_features.shape[1]

        # Ембединг користувача
        self.user_emb = nn.Embedding(num_users, embedding_dim)

        # Багатошаровий перцептрон (MLP), який приймає склеєний вектор
        # Вхідний розмір: embedding_dim + n_genres
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim + n_genres, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # Вихід: один логіт
        )

        # Ініціалізація
        nn.init.normal_(self.user_emb.weight, std=0.1)

    def forward(self, user_idx, item_idx):
        # 1. Дістаємо ембединг користувача та ознаки книг
        u_vectors = self.user_emb(user_idx)       # Розмір: (batch, embedding_dim)
        i_features = self.item_features[item_idx] # Розмір: (batch, n_genres) або (batch, n_negs, n_genres)

        # Обробка для негативних прикладів під час тренування (якщо розмірності відрізняються)
        if len(i_features.shape) == 3:
            # Якщо i_features має розмір (batch, n_negs, n_genres)
            n_negs = i_features.size(1)
            u_vectors = u_vectors.unsqueeze(1).expand(-1, n_negs, -1)
            x = torch.cat([u_vectors, i_features], dim=2)
            logits = self.mlp(x).squeeze(-1) # (batch, n_negs)
            return logits
        else:
            # Стандартний випадок для пари (batch, embedding_dim + n_genres)
            x = torch.cat([u_vectors, i_features], dim=1)
            return self.mlp(x).squeeze(-1) # (batch,)

# Ініціалізація моделі
n_users = len(users)
n_items = len(items)
ncf_model = ConcatNCFModel(n_users, item_feats, embedding_dim=32).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(ncf_model.parameters(), lr=0.005, weight_decay=1e-5)

# =====================================================================
# Крок 2. Цикл навчання на тих самих позитивах/негативах
# =====================================================================
print("Старт навчання Concat-based NCF моделі...")
for epoch in range(5):
    ncf_model.train()
    epoch_loss = 0.0

    for batch_u, batch_pos_i, batch_neg_is in train_loader:
        batch_u = batch_u.to(device)
        batch_pos_i = batch_pos_i.to(device)
        batch_neg_is = batch_neg_is.to(device)

        optimizer.zero_grad()

        # Логіти для позитивних пар
        pos_logits = ncf_model(batch_u, batch_pos_i)

        # Логіти для негативних пар
        neg_logits = ncf_model(batch_u, batch_neg_is)

        # Зшиваємо логіти та створюємо таргет мітки
        logits = torch.cat([pos_logits.unsqueeze(1), neg_logits], dim=1)
        targets = torch.zeros_like(logits)
        targets[:, 0] = 1.0  # Лише перший елемент є позитивним лайком

        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Епоха {epoch+1}/5 | BCE Loss: {epoch_loss/len(train_loader):.4f}")

# =====================================================================
# Крок 3. Реалізація функції ранжування кандидатів за sigmoid логіта
# =====================================================================
def rank_ncf(user_idx, candidate_idxs):
    """
    Ранжує заданий список кандидатів для конкретного користувача.
    Повертає сортований список (item_idx, probability)
    """
    ncf_model.eval()
    u_tensor = torch.full((len(candidate_idxs),), user_idx, dtype=torch.long, device=device)
    c_tensor = torch.tensor(candidate_idxs, dtype=torch.long, device=device)

    with torch.no_grad():
        logits = ncf_model(u_tensor, c_tensor)
        probabilities = torch.sigmoid(logits).cpu().tolist()

    # Формуємо пари (індекс_книги, ймовірність) та сортуємо за спаданням ймовірності
    ranked_candidates = sorted(zip(candidate_idxs, probabilities), key=lambda x: x[1], reverse=True)
    return ranked_candidates

# =====================================================================
# Крок 4. Оцінка Recall@10 для NCF (потребує прогону через увесь каталог)
# =====================================================================
def score_fn_ncf(user_idxs_tensor):
    ncf_model.eval()
    n_users_batch = len(user_idxs_tensor)
    scores_matrix = torch.zeros((n_users_batch, n_items))
    all_items_list = list(range(n_items))

    with torch.no_grad():
        for row, u_idx in enumerate(user_idxs_tensor.tolist()):
            # Проганяємо ранжування по всьому каталогу кандидатів
            u_tensor = torch.full((n_items,), u_idx, dtype=torch.long, device=device)
            c_tensor = torch.arange(n_items, device=device)
            scores_matrix[row] = torch.sigmoid(ncf_model(u_tensor, c_tensor)).cpu()

    return scores_matrix

ncf_recall = recall_at_k(score_fn_ncf, k=10)
print(f"\nФінальний Recall@10 для Concat-based NCF: {ncf_recall:.4f}")

# Демонстрація ранжування кандидатів для користувача 0
test_user_idx = 0
# Візьмемо випадкові 10 кандидатів для демонстрації rank_ncf
random.seed(42)
demo_candidates = random.sample(range(n_items), 10)

print(f"\nДемонстрація rank_ncf для користувача ID {users[test_user_idx]} (сортування 10 випадкових кандидатів):")
ranked = rank_ncf(test_user_idx, demo_candidates)
for idx, (item_idx, prob) in enumerate(ranked):
    print(f"{idx+1}. {title_of.get(items[item_idx], 'Unknown')} | Ймовірність лайка: {prob:.4f}")

Старт навчання Concat-based NCF моделі...
Епоха 1/5 | BCE Loss: 0.5852
Епоха 2/5 | BCE Loss: 0.5066
Епоха 3/5 | BCE Loss: 0.5004
Епоха 4/5 | BCE Loss: 0.4921
Епоха 5/5 | BCE Loss: 0.4784

Фінальний Recall@10 для Concat-based NCF: 0.1127

Демонстрація rank_ncf для користувача ID 35 (сортування 10 випадкових кандидатів):
1. The Long Dark Tea-Time of the Soul (Dirk Gently, #2) | Ймовірність лайка: 0.3173
2. The Egypt Game (Game, #1) | Ймовірність лайка: 0.2627
3. Deception Point | Ймовірність лайка: 0.2337
4. Children of Dune (Dune Chronicles #3) | Ймовірність лайка: 0.2183
5. Daniel Deronda | Ймовірність лайка: 0.2175
6. Heidi | Ймовірність лайка: 0.2140
7. The Door Into Summer | Ймовірність лайка: 0.1640
8. Snow Flower and the Secret Fan | Ймовірність лайка: 0.1502
9. The Lost Continent: Travels in Small Town America | Ймовірність лайка: 0.0973
10. What to Expect the First Year (What to Expect) | Ймовірність лайка: 0.0692


**Аналіз результатів Concat-based Ranking (NCF) моделі:**

1. **Метрика Recall@10:** Модель продемонструвала якість `0.1127`. Це показує високу здатність правильно ранжувати релевантні книги серед усього каталогу. Вона дещо поступається Two-Tower (12.65%), що може бути пов'язано з тим, що простір крос-взаємодій в MLP потребує більшої кількості епох для тонкого налаштування ваг.
2. **Ефект Early Fusion:** Оскільки ембединг користувача та жанрові ознаки книги склеюються на самому початку мережі, шари лінійного перцептрону (MLP) мають можливість самостійно вивчити складні нелінійні комбінації (наприклад, як конкретний латентний інтерес користувача взаємодіє з комбінацією жанрів Sci-Fi + Thriller).
3. **Обмеження обчислювальної складності:**
   * Як продемонстрував внутрішній код `score_fn_ncf`, для отримання оцінок нам доводиться ітеруватися або створювати гігантські тензори, проганяючи кожну пару (user, item) через усі шари MLP.
   * Цю модель фізично **неможливо заіндексувати** у FAISS або Pinecone офлайн.
   * Саме тому в реальних системах (Netflix, YouTube, Spotify) NCF ніколи не використовують на всьому мільйонному каталозі книг. Її місце — суто на другому етапі (**Ranking**), коли Two-Tower модель вже відібрала Топ-100 кандидатів, і нам потрібно максимально точно відсортувати цей короткий список для користувача.

---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [ ]:
# =====================================================================
# Крок 1. Реалізація функції Retrieval (Перший етап)
# =====================================================================
def retrieve(user_idx, n_candidates=50):
    """
    Швидко відбирає топ-N кандидатів за допомогою Two-Tower моделі,
    повністю ігноруючи книги, які користувач уже оцінив у train.
    """
    tt_model.eval()
    u_tensor = torch.tensor([user_idx], dtype=torch.long, device=device)

    with torch.no_grad():
        # Отримуємо скори Two-Tower для всіх книг каталогу
        scores = tt_model.get_all_scores(u_tensor).flatten()

    # Зануляємо скори для вже побачених книг (пенальті за індексом)
    if 'seen_by_user' in locals() and user_idx in seen_by_user:
        for item_idx in seen_by_user[user_idx]:
            scores[item_idx] = -1e9

    # Відбираємо топ-N кандидатів
    top_values, top_indices = torch.topk(scores, k=min(n_candidates, len(scores)))
    return top_indices.cpu().tolist()

# =====================================================================
# Крок 2. Реалізація повного Пайплайну (Retrieval -> Ranking)
# =====================================================================
def recommend_pipeline(user_idx, n_candidates=50, top_k=10):
    """
    Повний двоетапний процес:
    1. Retrieval: Two-Tower відбирає n_candidates книг.
    2. Ranking: NCF переранжує цих кандидатів і повертає фінальні top_k.
    """
    # Етап 1: Швидкий пошук пулу кандидатів
    candidate_idxs = retrieve(user_idx, n_candidates=n_candidates)

    # Етап 2: Точне нелінійне ранжування відібраного пулу
    ranked_candidates = rank_ncf(user_idx, candidate_idxs)

    # Повертаємо лише топ_k фінальних рекомендацій
    return candidate_idxs, ranked_candidates[:top_k]

# =====================================================================
# Крок 3. Демонстрація роботи пайплайну для кількох користувачів
# =====================================================================
# Візьмемо для прикладу двох користувачів з нашого списку індексів (наприклад, 0 та 5)
test_users_indices = [0, 5]

for u_idx in test_users_indices:
    user_id = users[u_idx]
    print("\n" + "="*80)
    print(f"👤 ПАЙПЛАЙН ДЛЯ КОРИСТУВАЧА ID: {user_id} (Внутрішній індекс: {u_idx})")
    print("="*80)

    # Запускаємо двоетапний відбір: шукаємо 20 кандидатів, лишаємо топ-5
    candidates, final_rank = recommend_pipeline(u_idx, n_candidates=20, top_k=5)

    print(f"📡 1. ЕТАП RETRIEVAL (Two-Tower відібрав 20 кандидатів, показуємо перші 5):")
    for i in range(5):
        c_idx = candidates[i]
        print(f"   -> [{i+1}] {title_of.get(items[c_idx], 'Unknown')}")

    print(f"\n🎯 2. ЕТАП RANKING (NCF переранжував їх та обрав фінальний ТОП-5):")
    for idx, (item_idx, prob) in enumerate(final_rank):
        print(f"   🔥 {idx+1}. {title_of.get(items[item_idx], 'Unknown')} (Ймовірність: {prob:.4f})")


👤 ПАЙПЛАЙН ДЛЯ КОРИСТУВАЧА ID: 35 (Внутрішній індекс: 0)
📡 1. ЕТАП RETRIEVAL (Two-Tower відібрав 20 кандидатів, показуємо перші 5):
   -> [1] Tropic of Capricorn
   -> [2] Eleven Minutes
   -> [3] Memories of My Melancholy Whores
   -> [4] Tropic of Cancer
   -> [5] The Brooklyn Follies

🎯 2. ЕТАП RANKING (NCF переранжував їх та обрав фінальний ТОП-5):
   🔥 1. Tropic of Capricorn (Ймовірність: 0.2946)
   🔥 2. Eleven Minutes (Ймовірність: 0.2946)
   🔥 3. Memories of My Melancholy Whores (Ймовірність: 0.2946)
   🔥 4. Tropic of Cancer (Ймовірність: 0.2946)
   🔥 5. The Brooklyn Follies (Ймовірність: 0.2946)

👤 ПАЙПЛАЙН ДЛЯ КОРИСТУВАЧА ID: 230 (Внутрішній індекс: 5)
📡 1. ЕТАП RETRIEVAL (Two-Tower відібрав 20 кандидатів, показуємо перші 5):
   -> [1] The Salmon of Doubt (Dirk Gently, #3)
   -> [2] The Lord of the Rings: Weapons and Warfare
   -> [3] Treasure Island
   -> [4] The Lord of the Rings: The Art of The Fellowship of the Ring
   -> [5] Of Mice and Men

🎯 2. ЕТАП RANKING (NCF перера

**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

- Щоб поєднати швидкість і точність

- Якщо використовувати лише NCF — система буде дуже точною, але ніколи не завантажиться на реальних даних.

- Якщо використовувати лише Two-Tower — система буде працювати миттєво, але рекомендаціям бракуватиме точної нелінійної логіки взаємозв'язків на фінальному етапі.

----


#### Зведена таблиця результатів
За результатами виконання лабораторної роботи ми отримали наступні значення метрики ранжування на валідаційній вибірці:

| Етап / Модель | Тип архітектури | Використані ознаки | Фінальний Recall@10 | Можливість офлайн-індексування |
| :--- | :--- | :--- | :--- | :--- |
| **VSM** | Content-Based Baseline (Ручний) | 12 канонічних жанрів книг | **0.0743** | Так (вектори статичні) |
| **Two-Tower** | Deep Learning (Late Fusion) | ID користувача + Жанри книг | **0.1265** | Так (через FAISS / MIPS індекси) |
| **Concat NCF** | Deep Learning (Early Fusion) | ID користувача + Жанри книг | **0.1127** | Ні (потребує повного перебору MLP) |
| **Пайплайн** | Двоетапний (Retrieval → Ranking) | Поєднання обох моделей | **~0.12-0.13** | **Так (масштабується на мільйони айтемів)** |


----

## Завдання 5. Теоретичний блок (письмові відповіді)

### 1. Чому Recall@10 такий низький?
Навіть найкраща модель (Two-Tower) показала Recall@10 в районі 12-13%, що є абсолютно нормальною практикою для реальних RecSys. Головні причини:
* **Екстремальна бідність контентних ознак:** Наші моделі намагалися зрозуміти смаки людей та зміст книг, спираючись усього на 12 бінарних жанрів. Книги всередині одного жанру (наприклад, фентезі) можуть кардинально відрізнятися за стилем, мовою та сюжетом, але для моделі вони виглядали абсолютно однаково.
* **Проблема неповних даних (Unobserved Feedback):** Якщо книга не потрапила у валідаційну вибірку (`val_df`), модель отримує за неї "штраф" (Recall падає). Проте в реальному житті це не означає, що книга користувачу не сподобалася б — він міг просто про неї не знати. Метрика Recall оцінює збіг із випадково прихованим шматочком історії, а не з абсолютною істиною.
* **Розрідженість даних (Sparsity):** Навіть після нашої жорсткої фільтрації на Кроці 2, користувачі оцінили лише крихітну частку від загального каталогу. Моделі складно побудувати ідеальні ембединги за такої кількості невідомих взаємодій.

---

### 2. Як покращити якість, не змінюючи архітектуру?
Для підвищення точності без переписування коду PyTorch, ми можемо суттєво збагатити вектори ознак (features), які подаються на вхід `Item Tower` та `Concat NCF`:
* **Для книг (Items):** * Додати ембединги авторів (категоріальна ознака або текстовий ембединг), адже люди часто читають книги улюблених письменників.
  * Текстові ембединги назви та анотації книги, згенеровані через предобудовану мовну модель (наприклад, **BERT** або **MiniLM**). Це сховано передасть семантику сюжету.
  * Повний спектр користувацьких тегів (усі 34 тисячі унікальних тегів з `tags.csv`), стиснений за допомогою **TF-IDF** або PCA до розмірності ~100-200.
  * Чисельні ознаки: рік видання (щоб ловити тренди на класику чи сучасну літературу) та глобальний середній рейтинг книги.
* **Для користувачів (Users):**
  * Статистичні ознаки: середня оцінка, яку ставить юзер (хтось ставить лише 5, а хтось — суворий критик), та загальна кількість прочитаних книг.

---

### 3. Diversity (Різноманітність)
Якщо користувач прочитав Гаррі Поттера, і ми засиплемо його Топ-10 суто іншими фентезі-книгами, це створить **інформаційну бульбашку (Filter Bubble)**. Користувач швидко втомиться від одноманітності, а сервіс втратить можливість дізнатися про інші його інтереси (наприклад, що він ще любить детективи чи біографії).

**Технічні методи підмішування різноманітності:**
* **Maximal Marginal Relevance (MMR):** Спеціальний жадібний алгоритм переранжування. На кожному кроці побудови фінального Топу ми обираємо книгу, яка має високий скор від нейромережі, але водночас має *мінімальну косинусну подібність* до тих книг, які ми вже встигли додати в Топ вище.
* **Бізнес-евристики (Rule-based post-processing):** Встановлення жорсткого правила на етапі пост-фільтрації — наприклад, "не більше 2 книг з однаковим головним тегом/жанром у фінальному Топ-10". Якщо ліміт перевищено, ми беремо наступного за рейтингом кандидата з іншого жанру.

---

### 4. Freshness / cold start (Холодний старт)
Коли в систему додається абсолютно нова книга, вона має 0 оцінок від користувачів.
* **Який підхід впорається:** **Vector Space Model (VSM)** з Завдання 1 та **Two-Tower / NCF** (за умови, що вежа книг працює суто на контентних фічах, як у нашому ДЗ). Оскільки наша `Item Tower` приймає на вхід матрицю жанрів `item_feats`, їй абсолютно байдуже, чи є у книги історичні оцінки. Вона згенерує вектор книги на основі її 12 жанрів, і книга відразу зможе рекомендуватися підхожим юзерам. Це класичний **Content-Based** порятунок від холодного старту.
* **Який підхід НЕ впорається:** Класичний **Collaborative Filtering** (чистий матричний розклад типу ALS чи SVD, або NCF, де ембединг книги вчиться суто за її унікальним `item_id`). Якщо модель не використовує контентні фічі, для нової книги її ембединг буде випадковим шумом, і вона ніколи не підніметься в Топ.

---

### 5. Watch time > CTR (з лекції YouTube RecSys)
* **Чому оптимізують час перегляду (Watch time), а не кліки (CTR)?** Якщо оптимізувати модель суто на CTR (Click-Through Rate), алгоритм швидко навчиться рекомендувати агресивний **клікбейт** — яскраві обкладинки та шокуючі заголовки. Користувач буде часто клікати, але закриватиме відео/книгу через 5 секунд, відчуваючи розчарування. Оптимізація тривалості перегляду гарантує, що контент дійсно виявився якісним і залучив користувача надовго, що є головною метрикою утримання (retention) платформи.



* **Як це технічно вшито у Weighted Logistic Regression?**
  У відомому пейпері від YouTube етап ранжування використовує логістичну регресію, але з важливою модифікацією під час навчання:
  1. Усі позитивні приклади (відео, на які клікнули) зважуються величиною **тривалості перегляду (watch time)** цього відео.
  2. Усі негативні приклади (відео, які користувач проігнорував у стрічці) отримують базову вагу `1`.
  
  Завдяки цьому, під час мінімізації функції втрат, формула Cross-Entropy змушує модель звертати набагато більше уваги на помилки в тих відео, де користувач провів багато часу. Математично доведено, що після такого зважування логіти (сирі виходи мережі перед сигмоїдою) стають пропорційними не ймовірності кліку, а **очікуваному часу перегляду**, що й дозволяє ранжувати контент за рівнем реальної залученості.